In [1]:
import csv
import hashlib
import json
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageOps


# ============================================================
# USER CONFIG (edit these paths/values directly on Kaggle)
# ============================================================
DATASET_A_ROOT = Path("/kaggle/input/datasets/cduytrn2/wildlife-dataset-part1")
DATASET_B_ROOT = Path("/kaggle/input/datasets/cduytrn2/vietnam-wildlife-dataset")
OUTPUT_ROOT = Path("/kaggle/working/wildlife_dataset_processed")

SEED = 42
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Train balancing policy (your requested default):
# 1) Split first
# 2) Find largest class size in TRAIN
# 3) Only augment smaller classes up to that size
# 4) Never downsample or augment the largest class
BALANCE_UPSAMPLE_ONLY = True
BALANCE_REFERENCE = "max_train"  # options: max_train, fixed
FIXED_TARGET = 160  # used only if BALANCE_REFERENCE = "fixed"
MAX_TARGET_CAP = 0  # 0 means no cap

# Output safety
RESET_OUTPUT = True

VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


@dataclass
class ImageItem:
    class_name: str
    src_path: Path
    source_name: str
    md5: str


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def resolve_image_root(root: Path) -> Path:
    root = root.resolve()

    def class_like_dir_count(candidate: Path) -> int:
        if not candidate.exists() or not candidate.is_dir():
            return 0
        count = 0
        for d in [p for p in candidate.iterdir() if p.is_dir()]:
            has_image = False
            for p in d.rglob("*"):
                if p.is_file() and p.suffix.lower() in VALID_EXTS:
                    has_image = True
                    break
            if has_image:
                count += 1
        return count

    candidates = []
    if (root / "dataset").exists():
        candidates.append(root / "dataset")
    candidates.append(root)

    best_candidate = None
    best_score = -1
    for candidate in candidates:
        score = class_like_dir_count(candidate)
        if score > best_score:
            best_score = score
            best_candidate = candidate

    if best_candidate is None or best_score <= 0:
        raise FileNotFoundError(f"No class folders found under: {root}")

    return best_candidate


def normalize_class_name(name: str) -> str:
    return name.strip().replace(" ", "_")


def iter_image_paths(class_dir: Path) -> List[Path]:
    files = []
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            files.append(p)
    return sorted(files)


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def collect_items(
    image_root: Path, source_name: str
) -> Tuple[List[ImageItem], Dict[str, int]]:
    items: List[ImageItem] = []
    class_counts: Dict[str, int] = {}

    for class_dir in sorted([p for p in image_root.iterdir() if p.is_dir()]):
        class_name = normalize_class_name(class_dir.name)
        image_paths = iter_image_paths(class_dir)
        class_counts[class_name] = len(image_paths)

        for img_path in image_paths:
            try:
                digest = md5_file(img_path)
            except Exception:
                continue
            items.append(
                ImageItem(
                    class_name=class_name,
                    src_path=img_path,
                    source_name=source_name,
                    md5=digest,
                )
            )

    return items, class_counts


def dedupe_items(
    items: Sequence[ImageItem],
) -> Tuple[Dict[str, List[ImageItem]], Dict[str, int]]:
    by_hash: Dict[str, List[ImageItem]] = {}
    for item in items:
        by_hash.setdefault(item.md5, []).append(item)

    ambiguous_hashes = set()
    dropped_same_class_duplicates = 0

    for digest, bucket in by_hash.items():
        classes = {x.class_name for x in bucket}
        if len(classes) > 1:
            ambiguous_hashes.add(digest)
        elif len(bucket) > 1:
            dropped_same_class_duplicates += len(bucket) - 1

    class_to_items: Dict[str, List[ImageItem]] = {}
    for digest, bucket in by_hash.items():
        if digest in ambiguous_hashes:
            continue
        first = bucket[0]
        class_to_items.setdefault(first.class_name, []).append(first)

    for cls in class_to_items:
        class_to_items[cls] = sorted(class_to_items[cls], key=lambda x: str(x.src_path))

    stats = {
        "total_input_images": len(items),
        "unique_hashes": len(by_hash),
        "ambiguous_hashes_cross_class": len(ambiguous_hashes),
        "dropped_same_class_duplicates": dropped_same_class_duplicates,
        "kept_images": sum(len(v) for v in class_to_items.values()),
        "num_classes": len(class_to_items),
    }
    return class_to_items, stats


def split_counts(n: int, val_ratio: float, test_ratio: float) -> Tuple[int, int, int]:
    if n < 3:
        return n, 0, 0

    val_n = max(1, int(round(n * val_ratio)))
    test_n = max(1, int(round(n * test_ratio)))

    if val_n + test_n >= n:
        val_n = 1
        test_n = 1

    train_n = n - val_n - test_n
    if train_n <= 0:
        train_n = max(1, n - 2)
        val_n = 1 if n >= 3 else 0
        test_n = 1 if n >= 3 else 0

    return train_n, val_n, test_n


def copy_items(
    items: Sequence[ImageItem], split_dir: Path, class_name: str, prefix: str
) -> List[Path]:
    class_dir = split_dir / class_name
    ensure_dir(class_dir)
    out_paths: List[Path] = []

    for idx, item in enumerate(items):
        ext = (
            item.src_path.suffix.lower()
            if item.src_path.suffix.lower() in VALID_EXTS
            else ".jpg"
        )
        out_name = f"{prefix}_{idx:05d}{ext}"
        dst = class_dir / out_name
        shutil.copy2(item.src_path, dst)
        out_paths.append(dst)

    return out_paths


def compute_md5_folder(folder: Path) -> Dict[str, str]:
    out = {}
    if not folder.exists():
        return out

    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            try:
                out[str(p)] = md5_file(p)
            except Exception:
                continue

    return out


def leakage_check(output_root: Path) -> Dict[str, int]:
    train_hashes = set(compute_md5_folder(output_root / "splits" / "train").values())
    val_hashes = set(compute_md5_folder(output_root / "splits" / "val").values())
    test_hashes = set(compute_md5_folder(output_root / "splits" / "test").values())

    return {
        "train_val_overlap": len(train_hashes.intersection(val_hashes)),
        "train_test_overlap": len(train_hashes.intersection(test_hashes)),
        "val_test_overlap": len(val_hashes.intersection(test_hashes)),
    }


def safe_open_rgb(path: Path) -> Optional[Image.Image]:
    try:
        with Image.open(path) as img:
            return img.convert("RGB")
    except Exception:
        return None


def augment_image(img: Image.Image, rng: random.Random) -> Image.Image:
    out = img.copy()

    if rng.random() < 0.5:
        out = ImageOps.mirror(out)

    if rng.random() < 0.2:
        angle = rng.uniform(-8.0, 8.0)
        out = out.rotate(angle, resample=Image.BILINEAR, expand=False, fillcolor=None)

    if rng.random() < 0.25:
        w, h = out.size
        scale = rng.uniform(0.85, 1.0)
        nw, nh = int(w * scale), int(h * scale)
        if nw > 0 and nh > 0:
            left = rng.randint(0, max(0, w - nw))
            top = rng.randint(0, max(0, h - nh))
            out = out.crop((left, top, left + nw, top + nh)).resize(
                (w, h), Image.BILINEAR
            )

    if rng.random() < 0.35:
        out = ImageEnhance.Brightness(out).enhance(rng.uniform(0.9, 1.1))
        out = ImageEnhance.Contrast(out).enhance(rng.uniform(0.9, 1.1))
        out = ImageEnhance.Color(out).enhance(rng.uniform(0.9, 1.1))

    if rng.random() < 0.1:
        out = out.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.2, 1.0)))

    if rng.random() < 0.1:
        arr = np.array(out).astype(np.float32)
        noise = np.random.normal(0, rng.uniform(2.0, 8.0), size=arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)

    return out


def write_csv(path: Path, rows: List[Dict], fieldnames: Sequence[str]) -> None:
    ensure_dir(path.parent)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def print_step(step_no: int, message: str) -> None:
    print(f"\n[Step {step_no}] {message}")


def print_human_summary(
    summary: Dict,
    dedupe_stats: Dict[str, int],
    leakage: Dict[str, int],
    split_rows: List[Dict],
    balance_rows: List[Dict],
) -> None:
    total_train = sum(int(row["train"]) for row in split_rows)
    total_val = sum(int(row["val"]) for row in split_rows)
    total_test = sum(int(row["test"]) for row in split_rows)
    total_aug = sum(int(row["augmented_added"]) for row in balance_rows)

    print("\n" + "=" * 66)
    print("PIPELINE SUMMARY (READABLE)")
    print("=" * 66)
    print(f"Dataset A root          : {summary['dataset_a_root']}")
    print(f"Dataset B root          : {summary['dataset_b_root']}")
    print(f"Output root             : {summary['output_root']}")
    print(f"Seed                    : {summary['seed']}")
    print(
        f"Split ratio             : train={1 - summary['val_ratio'] - summary['test_ratio']:.2f}, "
        f"val={summary['val_ratio']:.2f}, test={summary['test_ratio']:.2f}"
    )
    print(f"Number of classes       : {summary['num_classes_final']}")

    print("\nData merge & dedupe")
    print(f"- Total input images    : {dedupe_stats['total_input_images']}")
    print(f"- Unique hashes         : {dedupe_stats['unique_hashes']}")
    print(f"- Removed cross-class   : {dedupe_stats['ambiguous_hashes_cross_class']}")
    print(f"- Removed same-class dup: {dedupe_stats['dropped_same_class_duplicates']}")
    print(f"- Kept images           : {dedupe_stats['kept_images']}")

    print("\nSplit result")
    print(f"- Train images          : {total_train}")
    print(f"- Val images            : {total_val}")
    print(f"- Test images           : {total_test}")

    print("\nLeakage check")
    print(f"- Train vs Val overlap  : {leakage['train_val_overlap']}")
    print(f"- Train vs Test overlap : {leakage['train_test_overlap']}")
    print(f"- Val vs Test overlap   : {leakage['val_test_overlap']}")

    print("\nTrain balancing")
    print(f"- Policy                : upsample minority classes only")
    print(f"- Balance reference     : {summary['balance_reference']}")
    if summary["balance_reference"] == "fixed":
        print(f"- Fixed target          : {summary['fixed_target']}")
    print(f"- Actual target         : {summary['actual_balance_target']}")
    print(f"- Augmented images      : {total_aug}")

    print("\nReports written")
    print("- merge_source_report.csv")
    print("- merge_report.csv")
    print("- split_report.csv")
    print("- leakage_check.csv")
    print("- balance_report.csv")
    print("- pipeline_summary.json")
    print("=" * 66)


def choose_balance_target(train_counts: Dict[str, int]) -> int:
    if not train_counts:
        return FIXED_TARGET

    if BALANCE_REFERENCE == "max_train":
        target = max(train_counts.values())
    else:
        target = FIXED_TARGET

    if MAX_TARGET_CAP > 0:
        target = min(target, MAX_TARGET_CAP)

    return target


def balance_train_upsample_only(output_root: Path, seed: int) -> Tuple[List[Dict], int]:
    rng = random.Random(seed)
    np.random.seed(seed)

    train_dir = output_root / "splits" / "train"
    ensure_dir(train_dir)

    class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])
    train_counts: Dict[str, int] = {}
    for cls in class_names:
        imgs = [
            p
            for p in (train_dir / cls).glob("*")
            if p.is_file() and p.suffix.lower() in VALID_EXTS
        ]
        train_counts[cls] = len(imgs)

    target = choose_balance_target(train_counts)

    rows: List[Dict] = []
    for cls in class_names:
        cls_dir = train_dir / cls
        base_imgs = sorted(
            [
                p
                for p in cls_dir.glob("*")
                if p.is_file() and p.suffix.lower() in VALID_EXTS
            ]
        )
        before_n = len(base_imgs)
        aug_n = 0

        if before_n > 0 and BALANCE_UPSAMPLE_ONLY and before_n < target:
            need = target - before_n
            for i in range(need):
                src = base_imgs[i % before_n]
                img = safe_open_rgb(src)
                if img is None:
                    continue
                aug = augment_image(img, rng)
                out_path = cls_dir / f"aug_{i:05d}.jpg"
                try:
                    aug.save(out_path, quality=92)
                    aug_n += 1
                except Exception:
                    continue

        after_n = len(
            [
                p
                for p in cls_dir.glob("*")
                if p.is_file() and p.suffix.lower() in VALID_EXTS
            ]
        )
        rows.append(
            {
                "class_name": cls,
                "train_before": before_n,
                "augmented_added": aug_n,
                "train_after": after_n,
                "balance_target": target,
                "policy": "upsample_only",
            }
        )

    return rows, target


def run_pipeline() -> None:
    random.seed(SEED)
    np.random.seed(SEED)

    print_step(1, "Initialize output folders")

    output_root = OUTPUT_ROOT.resolve()
    if RESET_OUTPUT and output_root.exists():
        shutil.rmtree(output_root)

    reports_dir = output_root / "reports"
    ensure_dir(reports_dir)

    print_step(2, "Resolve dataset roots")
    root_a = resolve_image_root(DATASET_A_ROOT)
    root_b = resolve_image_root(DATASET_B_ROOT)
    print(f"- Dataset A: {root_a}")
    print(f"- Dataset B: {root_b}")

    print_step(3, "Collect image metadata and hashes")
    items_a, counts_a = collect_items(root_a, "dataset_a")
    items_b, counts_b = collect_items(root_b, "dataset_b")
    all_items = items_a + items_b
    print(f"- Dataset A images: {len(items_a)}")
    print(f"- Dataset B images: {len(items_b)}")
    print(f"- Total images    : {len(all_items)}")

    source_rows: List[Dict] = []
    all_classes = sorted(set(counts_a.keys()).union(counts_b.keys()))
    for cls in all_classes:
        source_rows.append(
            {
                "class_name": cls,
                "dataset_a_count": counts_a.get(cls, 0),
                "dataset_b_count": counts_b.get(cls, 0),
                "source_total": counts_a.get(cls, 0) + counts_b.get(cls, 0),
            }
        )
    write_csv(
        reports_dir / "merge_source_report.csv",
        source_rows,
        ["class_name", "dataset_a_count", "dataset_b_count", "source_total"],
    )

    print_step(4, "De-duplicate merged data")
    class_to_items, dedupe_stats = dedupe_items(all_items)
    print(f"- Classes after dedupe: {len(class_to_items)}")
    print(f"- Kept images         : {dedupe_stats['kept_images']}")

    merge_rows = []
    for cls in sorted(class_to_items.keys()):
        merge_rows.append(
            {
                "class_name": cls,
                "after_dedupe_count": len(class_to_items[cls]),
            }
        )
    write_csv(
        reports_dir / "merge_report.csv",
        merge_rows,
        ["class_name", "after_dedupe_count"],
    )

    train_dir = output_root / "splits" / "train"
    val_dir = output_root / "splits" / "val"
    test_dir = output_root / "splits" / "test"
    ensure_dir(train_dir)
    ensure_dir(val_dir)
    ensure_dir(test_dir)

    split_rows: List[Dict] = []
    split_paths: Dict[str, Dict[str, List[Path]]] = {"train": {}, "val": {}, "test": {}}

    print_step(5, "Split data into train/val/test")
    rng = random.Random(SEED)
    for cls, items in sorted(class_to_items.items()):
        local = list(items)
        rng.shuffle(local)

        n_train, n_val, n_test = split_counts(len(local), VAL_RATIO, TEST_RATIO)
        train_items = local[:n_train]
        val_items = local[n_train : n_train + n_val]
        test_items = local[n_train + n_val : n_train + n_val + n_test]

        train_paths = copy_items(train_items, train_dir, cls, "orig")
        val_paths = copy_items(val_items, val_dir, cls, "orig")
        test_paths = copy_items(test_items, test_dir, cls, "orig")

        split_paths["train"][cls] = train_paths
        split_paths["val"][cls] = val_paths
        split_paths["test"][cls] = test_paths

        split_rows.append(
            {
                "class_name": cls,
                "total_after_dedupe": len(local),
                "train": len(train_paths),
                "val": len(val_paths),
                "test": len(test_paths),
            }
        )

    write_csv(
        reports_dir / "split_report.csv",
        split_rows,
        ["class_name", "total_after_dedupe", "train", "val", "test"],
    )
    print(f"- Split done for {len(split_rows)} classes")

    print_step(6, "Run leakage check")
    leakage = leakage_check(output_root)
    write_csv(
        reports_dir / "leakage_check.csv",
        [leakage],
        ["train_val_overlap", "train_test_overlap", "val_test_overlap"],
    )
    print(
        "- Overlap counts: "
        f"train-val={leakage['train_val_overlap']}, "
        f"train-test={leakage['train_test_overlap']}, "
        f"val-test={leakage['val_test_overlap']}"
    )

    print_step(7, "Balance train set by upsampling minority classes")
    balance_rows, balance_target = balance_train_upsample_only(output_root, SEED)
    write_csv(
        reports_dir / "balance_report.csv",
        balance_rows,
        [
            "class_name",
            "train_before",
            "augmented_added",
            "train_after",
            "balance_target",
            "policy",
        ],
    )
    total_aug = sum(int(row["augmented_added"]) for row in balance_rows)
    print(f"- Balance target : {balance_target}")
    print(f"- Augmented total: {total_aug}")

    summary = {
        "dataset_a_root": str(root_a),
        "dataset_b_root": str(root_b),
        "output_root": str(output_root),
        "seed": SEED,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
        "balance_upsample_only": BALANCE_UPSAMPLE_ONLY,
        "balance_reference": BALANCE_REFERENCE,
        "fixed_target": FIXED_TARGET,
        "max_target_cap": MAX_TARGET_CAP,
        "actual_balance_target": balance_target,
        "dedupe_stats": dedupe_stats,
        "leakage": leakage,
        "num_classes_final": len(class_to_items),
    }

    with (reports_dir / "pipeline_summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print_step(8, "Print final readable summary")
    print_human_summary(summary, dedupe_stats, leakage, split_rows, balance_rows)
    print("Done.")


if __name__ == "__main__":
    run_pipeline()



[Step 1] Initialize output folders

[Step 2] Resolve dataset roots
- Dataset A: /kaggle/input/datasets/cduytrn2/wildlife-dataset-part1/dataset
- Dataset B: /kaggle/input/datasets/cduytrn2/vietnam-wildlife-dataset/dataset

[Step 3] Collect image metadata and hashes
- Dataset A images: 32495
- Dataset B images: 76553
- Total images    : 109048

[Step 4] De-duplicate merged data
- Classes after dedupe: 570
- Kept images         : 108560

[Step 5] Split data into train/val/test
- Split done for 570 classes

[Step 6] Run leakage check
- Overlap counts: train-val=0, train-test=0, val-test=0

[Step 7] Balance train set by upsampling minority classes
- Balance target : 160
- Augmented total: 4414

[Step 8] Print final readable summary

PIPELINE SUMMARY (READABLE)
Dataset A root          : /kaggle/input/datasets/cduytrn2/wildlife-dataset-part1/dataset
Dataset B root          : /kaggle/input/datasets/cduytrn2/vietnam-wildlife-dataset/dataset
Output root             : /kaggle/working/wildlife_da

In [2]:
from pathlib import Path
import pandas as pd

# ===== Config giống pipeline đã chạy =====
OUTPUT_ROOT = Path("/kaggle/working/wildlife_dataset_processed")
TRAIN_DIR = OUTPUT_ROOT / "splits" / "train"
REPORTS_DIR = OUTPUT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"Không thấy thư mục train: {TRAIN_DIR}")

rows = []
for class_dir in sorted([p for p in TRAIN_DIR.iterdir() if p.is_dir()]):
    files = [p for p in class_dir.glob("*") if p.is_file() and p.suffix.lower() in VALID_EXTS]
    orig_count = sum(1 for p in files if p.name.startswith("orig_"))
    aug_count = sum(1 for p in files if p.name.startswith("aug_"))
    total_count = len(files)

    rows.append({
        "class_name": class_dir.name,
        "train_current_total": total_count,
        "original_count": orig_count,
        "augmented_count": aug_count,
    })

df = pd.DataFrame(rows).sort_values(["augmented_count", "train_current_total"], ascending=[False, False])

# Lưu CSV thống kê theo yêu cầu
out_csv = REPORTS_DIR / "train_class_stats_with_augment.csv"
df.to_csv(out_csv, index=False, encoding="utf-8")

# ===== In tổng hợp dễ đọc =====
total_classes = len(df)
augmented_classes = int((df["augmented_count"] > 0).sum())

print("=" * 72)
print("THONG KE TRAIN HIEN TAI")
print("=" * 72)
print(f"So class trong train hien tai: {total_classes}")
print(f"So class da duoc augment / tong class: {augmented_classes} / {total_classes}")

print("\nTop 10 class duoc augment nhieu nhat:")
top10 = df.head(10).copy()
if len(top10) == 0:
    print("- Khong co class nao trong train.")
else:
    # In gọn đẹp
    print(top10[["class_name", "augmented_count", "original_count", "train_current_total"]].to_string(index=False))

print(f"\nDa luu CSV thong ke: {out_csv}")
print("=" * 72)

THONG KE TRAIN HIEN TAI
So class trong train hien tai: 570
So class da duoc augment / tong class: 302 / 570

Top 10 class duoc augment nhieu nhat:
             class_name  augmented_count  original_count  train_current_total
   Poliolimnas_cinereus               69              91                  160
 Urocissa_erythroryncha               66              94                  160
   Sinomicrurus_peinani               65              95                  160
           Boiga_cyanea               64              96                  160
      Oriolus_chinensis               64              96                  160
       Passer_flaveolus               64              96                  160
  Tephrodornis_virgatus               64              96                  160
     Microhyla_malcolmi               63              97                  160
  Acanthosaura_nataliae               61              99                  160
Heterophasia_desgodinsi               61              99                 